# Predicción de salarios

En este ejercicio ilustraremos el fenómeno de overfitting. Para ello investigaremos, desde una perspectiva predictiva, la determinación del salario de los trabajadores.

En lo que sigue, $Y$ es el log del salario por hora del trabajador y $W$ es un vector de características. Nos interesa la siguiente pregunta

- ¿Cómo podemos hacer uso de las características del trabajador para predecir de mejor manera los salarios?

## Datos

Usaremos datos de la GEIH para Barranquilla, año 2024. Seleccionamos únicamente trabajadores asalariados del sector privado, entre 25 y 64 años. La variable de interés es el salario por hora

In [ ]:
suppressWarnings(
pacman::p_load(xtable, dplyr)
)

In [ ]:
file<-"https://raw.githubusercontent.com/andvarga-eco/econometrics_ml/refs/heads/main/data/geih_24_baq.csv"
data<-read.csv(file)

Tenemos las siguientes variables

-INGLABO: Ingresos laborales
- P6500: Antes de descuentos ¿cuánto ganó ... El mes pasado en este empleo?
- P6800: ¿cuántas horas a la semana trabaja normalmente ... En ese trabajo?
- RAMA2D_R4: Sector CIIU 2 dígitos Versión 4 AC
- RAMA4D_R4: Sector CIIU 4 dígitos Versión 4 AC
- OFICIO_C8: Oficio (https://ocupacol.mintrabajo.gov.co/Jerarquias/Jerarquias)
- P6430: En este trabajo es…(posición ocupacional): 1. obrero o empleado de empresa particular,...
- P6040: ¿Cuántos años cumplidos tiene?
- P3271 : ¿Cuál fue su sexo al nacer? 1: masculino, 2: femenino
- P3042: ¿Cuál es el mayor nivel educativo alcanzado y el último grado o semestre aprobado por …...? 1.Ninguno, 2.Preescolar, 
3.Básica primaria (1o - 5o), 4.Básica secundaria (6o - 9o), 5.Media académica (Bachillerato clásico), 6.Media técnica (Bachillerato técnico), 7.Normalista, 8.Técnica profesional, 9.Tecnológica, 10.Universitaria, 11.Especialización, 12.Maestría, 13.Doctorado, 99.No sabe, no informa

Creamos la variable $Y$

In [ ]:
data<-data|>filter(P6040 %in% 25:64)|>filter(P3042!=99)
data<-na.omit(data)

In [ ]:
data<-data|>filter(P6800<=60)|>mutate(w=INGLABO/(P6800*4),lw=log(w),
                                      edad2=P6040^2, edad3=P6040^3, edad4=P6040^4)

## Modelos

Usaremos dos especificaciones

- Básica: $X$ está dada por regresores crudos (género, edad, educación, industria)
- Flexible: $X$ contiene los regresores crudos ($W$) y un conjunto de transformaciones de la edad e interacciones de doble vía entre de la edad y sus polinomios con las demas variables. Una interacción sería por ejemplo $edad \times Universidad$

Para entender el problema del *overfitting* debemos estimar el modelo en una muestra de entrenamiento, y probarlo en una muestra de prueba

In [ ]:
# Partir los datos en data de entrenamiento (80%) y data de testeo (20%)
set.seed(1) # para poder replicar
n <- nrow(data)
random <- sample(1:n, floor(n * 4 / 5)) # draw (4/5)*n random numbers de 1 a n sin reemplazo
train <- data[random, ]
test <- data[-random, ]

## Estimación y ajuste dentro de muestra

In [ ]:
# Modelos
basic<-lw~(C(P3271)+P6040+C(P3042)+C(RAMA2D_R4))
flex<-lw~C(P3271)+C(P3042)+C(RAMA2D_R4)+(P6040+edad2+edad3+edad4)*(C(P3271)+C(P3042)+C(RAMA2D_R4))

In [ ]:
# 1. Modelo básico
regbasic <- lm(basic, data = train)
# número de regresores
cat("Number of regressors in the basic model:", length(regbasic$coef), "\n")

In [ ]:
# 2. Modelo flexible
regflex <- lm(flex, data = train)
# número de regresores
cat("Number of regressors in the flex model:", length(regflex$coef), "\n")

Evaluamos el desempeño predictivo dentro de muestra. Usamos dos medidas: $R^2$ y $MSE$

In [ ]:
sumbasic <- summary(regbasic)
sumflex <- summary(regflex)

ntrain <- nrow(train)

# R-squared and adjusted R-squared
r2_1 <- sumbasic$r.squared
cat("R-squared for the basic model: ", r2_1, "\n")
r2_adj1 <- sumbasic$adj.r.squared
cat("adjusted R-squared for the basic model: ", r2_adj1, "\n")

r2_2 <- sumflex$r.squared
cat("R-squared for the flexible model: ", r2_2, "\n")
r2_adj2 <- sumflex$adj.r.squared
cat("adjusted R-squared for the flexible model: ", r2_adj2, "\n")

mse1 <- mean(sumbasic$res^2)
cat("MSE for the basic model: ", mse1, "\n")
p1 <- sumbasic$df[1] # number of regressors
mse_adj1 <- (ntrain / (ntrain - p1)) * mse1
cat("adjusted MSE for the basic model: ", mse_adj1, "\n")

#MSE
mse2 <- mean(sumflex$res^2)
cat("MSE for the flexible model: ", mse2, "\n")
p2 <- sumflex$df[1]
mse_adj2 <- (ntrain / (ntrain - p2)) * mse2
cat("adjusted MSE for the lasso flexible model: ", mse_adj2, "\n")


In [ ]:
table <- matrix(0, 2, 5)
table[1, 1:5] <- c(p1, r2_1, mse1, r2_adj1, mse_adj1)
table[2, 1:5] <- c(p2, r2_2, mse2, r2_adj2, mse_adj2)
colnames(table) <- c("p", "$R^2_{sample}$", "$MSE_{sample}$", "$R^2_{adjusted}$", "$MSE_{adjusted}$")
rownames(table) <- c("basic reg", "flexible reg")
tab <- xtable(table, digits = c(0, 0, 2, 2, 2, 2))
tab

## Ajuste fuera de muestra

In [ ]:
library(dplyr)

test %>%
  anti_join(distinct(train, RAMA2D_R4), by = "RAMA2D_R4") %>%
  count(RAMA2D_R4, sort = TRUE)


In [ ]:
# 1. Modelo básico
options(warn = -1)
test_ad<-test|>filter(RAMA2D_R4!="30" & RAMA2D_R4!="3")
yhat_bas <- predict(regbasic, newdata = test_ad)
y_test <- test_ad$lw
mean_train <- mean(train$lw)
mse_test1 <- sum((y_test - yhat_bas)^2) / length(y_test)
r2_test1 <- 1 - mse_test1 / mean((y_test - mean_train)^2)

cat("Test MSE for the basic model: ", mse_test1, " ")
cat("Test R2 for the basic model: ", r2_test1)

In [ ]:
# 2. Modelo flexible

options(warn = -1)

yhat_flex <- predict(regflex, newdata = test_ad)
y_test <- test_ad$lw
mean_train <- mean(train$lw)
mse_test2 <- sum((y_test - yhat_flex)^2) / length(y_test)
r2_test2 <- 1 - mse_test2 / mean((y_test - mean_train)^2)

cat("Test MSE for the flexible model: ", mse_test2, " ")

cat("Test R2 for the flexible model: ", r2_test2)

Se observa claramente el fenómeno de sobreajuste con el modelo flexible. Dentro de muestra luce bien, pero fuera de muestra es pésimo

## Actividad

Repita el ejercicio, pero con una especificación extraflexible. Use como referencia la especificación usada en [Causal ML](https://colab.research.google.com/github/CausalAIBook/MetricsMLNotebooks/blob/main/PM1/r-ols-and-lasso-for-wage-prediction.irnb#scrollTo=U2lQCgJeyka0)